In [57]:
#import required libraries
import pandas  as pd 
import sqlite3 as sql

In [58]:
db_path="database/books.db"

#connecting to the database
connect=sql.connect(db_path)

print("Database Successfully Connected to SQLite")

Database Successfully Connected to SQLite


In [59]:
# Display all tables in the SQLite database
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    connect
)

tables

,name
0,category
1,books


***QUERIES :***




***1. Query books with a rating greater than or equal to 4***

In [60]:
query="select title,rating,price_inr from books where rating>=4"
result=pd.read_sql_query(query,connect)
print(result)

                                                title  rating  price_inr
0   Full Moon over Noah’s Ark: An Odyssey to Mount...       4    5214.86
1                    A Year in Provence (Provence #1)       4    6000.84
2                  1,000 Places to See Before You Die       5    2751.44
3                                       Sharp Objects       4    5045.01
4                                 The Past Never Ends       4    5960.75
5     The Murder of Roger Ackroyd (Hercule Poirot #4)       4    4652.55
6              A Time of Torment (Charlie Parker #14)       5    5100.92
7   Murder at the 42nd Street Library (Raymond Amb...       4    5734.98
8                         Private Paris (Private #10)       5    5022.85
9                        We Love You, Charlie Freeman       5    5303.48
10                                             Thirst       5    1821.98
11                                    The Vacationers       4    4446.82
12               The Regional Office Is Under Attac

***2. Display Unique rating values***

In [61]:
query = '''
SELECT DISTINCT rating
FROM books
ORDER BY rating ASC
'''

result = pd.read_sql_query(query, connect)

print(result)

   rating
0       1
1       2
2       3
3       4
4       5


***3. Display the top 5 most expensive books based on price_gbp.***

In [62]:
query="select book_id,title,price_gbp from books order by price_gbp DESC limit 5"
result=pd.read_sql_query(query,connect)
print(result)

   book_id                                              title  price_gbp
0       26                      Boar Island (Anna Pigeon #19)      59.48
1       72                                          The Stand      57.86
2       45  Immunity: How Elie Metchnikoff Changed the Cou...      57.36
3       54  The Disappearing Spoon: And Other True Tales o...      57.35
4        8                   A Year in Provence (Provence #1)      56.88


***4. Display the book ID, title, rating, and price of all books whose price is between £20 and £40, sorted by price from highest to lowest.***

In [63]:
query="select book_id,title,rating,price_gbp from books where price_gbp between 20 and 40 order by price_gbp DESC"
result=pd.read_sql_query(query,connect)
print(result)

    book_id                                              title  rating  \
0        58                                           Security       2   
1        10          Neither Here nor There: Travels in Europe       3   
2        47  Tipping Point for Planet Earth: How Close Are ...       1   
3         5                               Under the Tuscan Sun       3   
4        63                   Psycho: Sanitarium (Psycho #1.5)       5   
5         4  Vagabonding: An Uncommon Guide to the Art of L...       2   
6        66                                Dracula the Un-Dead       5   
7        24                                        Most Wanted       3   
8        69                                             Misery       2   
9        56                     Seven Brief Lessons on Physics       4   
10        7                           The Great Railway Bazaar       1   
11       57                                   The Selfish Gene       1   
12       49  Diary of a Citizen Scient

***5. Display the book title, category name, rating, and price of books that have a rating of 4 or higher. Sort the results first by category name alphabetically and then by price from highest to lowest.***

In [64]:
query='''
select b.title,c.category_name,b.rating,b.price_gbp
from books as b 
JOIN 
category as c 
on 
b.category_id=c.category_id
where b.rating>=4 
order by c.category_name ASC,b.price_gbp DESC'''
result=pd.read_sql_query(query,connect)
print(result)

                                                title category_name  rating  \
0                                               Shtum       Fiction       4   
1            Finders Keepers (Bill Hodges Trilogy #2)       Fiction       5   
2                               The Testament of Mary       Fiction       4   
3                The Regional Office Is Under Attack!       Fiction       5   
4                        We Love You, Charlie Freeman       Fiction       5   
5                         Private Paris (Private #10)       Fiction       5   
6                                     The Vacationers       Fiction       4   
7                                     The Time Keeper       Fiction       5   
8                                              Thirst       Fiction       5   
9                                        'Salem's Lot        Horror       4   
10                                     Needful Things        Horror       4   
11                   Psycho: Sanitarium (Psycho #1.5

***6.Display all books whose price_gbp is greater than the average price of all books. 
Show the book title, category name, price, and rating. Sort the results by price from highest to lowest.***


In [65]:
query = '''
SELECT 
    b.title,
    c.category_name,
    b.price_gbp,
    b.rating

FROM books AS b

JOIN category AS c
ON b.category_id = c.category_id

WHERE b.price_gbp > (
    SELECT AVG(price_gbp)
    FROM books
)

ORDER BY b.price_gbp DESC
'''

result = pd.read_sql_query(query, connect)

print(result)

                                                title category_name  \
0                       Boar Island (Anna Pigeon #19)       Mystery   
1                                           The Stand        Horror   
2   Immunity: How Elie Metchnikoff Changed the Cou...       Science   
3   The Disappearing Spoon: And Other True Tales o...       Science   
4                    A Year in Provence (Provence #1)        Travel   
5                                 The Past Never Ends       Mystery   
6   The Fabric of the Cosmos: Space, Time, and the...       Science   
7                                               Shtum       Fiction   
8   Murder at the 42nd Street Library (Raymond Amb...       Mystery   
9                      The Last Mile (Amos Decker #2)       Mystery   
10  The Murder That Never Was (Forensic Instincts #5)       Fiction   
11           Finders Keepers (Bill Hodges Trilogy #2)       Fiction   
12                              The Testament of Mary       Fiction   
13    

***7. Find the category or categories whose average book price is greater than the overall average book price. 
For those categories, display the category name, total number of books, average price, highest price, and lowest price. Sort the results by average price from highest to lowes***


In [66]:
query = '''
SELECT 
    c.category_name,
    COUNT(b.book_id) AS total_books,
    AVG(b.price_gbp) AS average_price,
    MAX(b.price_gbp) AS highest_price,
    MIN(b.price_gbp) AS lowest_price

FROM books AS b

JOIN category AS c
ON b.category_id = c.category_id

GROUP BY c.category_name

HAVING AVG(b.price_gbp) > (
    SELECT AVG(price_gbp)
    FROM books
)

ORDER BY average_price DESC
'''

result = pd.read_sql_query(query, connect)

print(result)

  category_name  total_books  average_price  highest_price  lowest_price
0       Fiction           16      41.428750          55.84         17.27
1        Travel           11      39.794545          56.88         23.21


***8. Find categories that have more than 10 books. Display the category name, total number of books, and average book price. Only include categories whose average rating is 3 or higher. Sort the results by total number of books from highest to lowest.***

In [67]:
query = '''
SELECT 
    c.category_name,
    COUNT(b.book_id) AS total_books,
    AVG(b.price_gbp) AS average_price

FROM books AS b

JOIN category AS c
ON b.category_id = c.category_id

GROUP BY c.category_name

HAVING COUNT(b.book_id) > 10
AND AVG(b.rating) >= 3

ORDER BY total_books DESC
'''
result=pd.read_sql_query(query,connect)
print(result)

  category_name  total_books  average_price
0       Fiction           16       41.42875


***9. Reproducing SQL JOIN Using Pandas merge()***

In [68]:
# Load the books table into a DataFrame
books_df = pd.read_sql_query(
    "SELECT * FROM books",
    connect
)

# Load the category table into a DataFrame
category_df = pd.read_sql_query(
    "SELECT * FROM category",
    connect
)

In [69]:
# Perform an inner join using category_id
merged_df = pd.merge(
    books_df,
    category_df,
    on="category_id",
    how="inner"
)

merged_df.head()

,book_id,title,price_gbp,rating,in_stock,price_inr,category_id,category_name
0,1,It's Only the Himalayas,45.17,2,1,4765.44,1,Travel
1,2,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,4,1,5214.86,1,Travel
2,3,See America: A Celebration of Our National Par...,48.87,3,1,5155.78,1,Travel
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,2,1,3897.17,1,Travel
4,5,Under the Tuscan Sun,37.33,3,1,3938.31,1,Travel


In [70]:
# SQL query to join books and category tables
sql_join = '''
SELECT 
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp

FROM books AS b

JOIN category AS c
ON b.category_id = c.category_id
'''

# Execute the SQL JOIN query
sql_result = pd.read_sql_query(sql_join, connect)

sql_result.head()

,title,category_name,rating,price_gbp
0,It's Only the Himalayas,Travel,2,45.17
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,4,49.43
2,See America: A Celebration of Our National Par...,Travel,3,48.87
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,2,36.94
4,Under the Tuscan Sun,Travel,3,37.33


In [71]:
# Select the same columns from the merged DataFrame
pandas_result = merged_df[
    [
        "title",
        "category_name",
        "rating",
        "price_gbp"
    ]
]

pandas_result.head()

,title,category_name,rating,price_gbp
0,It's Only the Himalayas,Travel,2,45.17
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,4,49.43
2,See America: A Celebration of Our National Par...,Travel,3,48.87
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,2,36.94
4,Under the Tuscan Sun,Travel,3,37.33


In [72]:
# Check whether both DataFrames contain the same data
sql_result.equals(pandas_result)

True

In [73]:
# Close the SQLite database connection
connect.close()

print("Database connection closed successfully!")

Database connection closed successfully!
